# Build a miniscope recording, one stage at a time

A 1-photon miniscope recording is the end of a physical chain: neurons fire →
calcium binds an indicator → light is emitted → optics blur and dim it → the
brain moves → a sensor counts noisy photons. The minian *analysis* pipeline runs
that chain **backwards** to recover the neurons. Here we run it **forwards** —
and we build it **together**, choosing the physics as we go.

Each stage has the same rhythm:

1. **Understand** — what this piece of physics is, and what to watch.
2. **Explore** — drag sliders and *see* the effect.
3. **Commit** — set the values you want, add the stage, and move on.

The recording grows as we add stages; by the end we have a full synthetic movie
*and* the exact ground truth behind it — positions, traces, footprints, motion,
and which cells are even physically recoverable. That truth is what Notebook 2
scores the real pipeline against.

> The interactive sliders need a **live kernel** — run this notebook (don't just
> read it). Run cells top to bottom; each stage uses the choices committed above
> it.

## Setup

The simulator is `minian.simulation`. We also pull in `mediapy` (smooth inline video playback) and a couple of tiny plotting helpers. `mediapy` finds `ffmpeg` on your PATH automatically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from ipywidgets import FloatSlider, HBox, VBox
from IPython.display import display
import mediapy

from minian.simulation import (
    Acquisition, Optics, ImageSensor, Tissue, Output, Spec, simulate,
    PlaceSomata, CellActivity, CellOptics, Render,
    Neuropil, Bleaching, BrainMotion, Vignette, Leakage, Sensor,
)

# Interactive backend: each widget builds ONE figure and updates it in place as
# you drag a slider -- smooth (no flicker) and immune to VS Code's duplicate-
# output bug (no Output widget / re-display). If plots ever do duplicate after
# reopening the notebook, run Command Palette -> "Developer: Reload Window".
%matplotlib widget
plt.ioff()  # don't auto-show figures; we display each canvas once, then mutate it
plt.rcParams["image.cmap"] = "magma"


def play(movie, fps=20, height=280, title=None):
    # normalize a (frame, h, w) movie to [0,1] and show a looping inline video
    arr = np.asarray(movie, dtype=float)
    lo, hi = float(arr.min()), float(arr.max())
    mediapy.show_video((arr - lo) / (hi - lo + 1e-9), fps=fps, height=height, title=title, codec="h264")


def interactive_panel(sliders, draw, canvas, ncols=2):
    # Wire every slider to redraw the SAME persistent canvas in place. `draw`
    # reads the slider values itself and mutates the figure; we never re-display.
    for s in sliders.values():
        if hasattr(s, "continuous_update"):  # sliders only; some widgets lack this trait
            s.continuous_update = False
        s.style.description_width = "104px"
        s.layout.width = "340px"
    for s in sliders.values():
        s.observe(lambda _change: draw(), names="value")
    draw()
    vals = list(sliders.values())
    per = -(-len(vals) // ncols)  # ceil
    display(HBox([VBox(vals[i * per:(i + 1) * per]) for i in range(ncols)]))
    display(canvas)

## Our recording, as we build it

Two things carry forward as we go:

- **`acq`** — the *scope*: optics, sensor, tissue, frame rate, duration. We set
  this first (Stage 1).
- **`steps`** — the ordered list of construction stages. We start empty and
  **append one stage per section** as we commit to it.

`build()` simulates the recording from whatever stages we've committed so far;
`preview()` does the same on a small, fast FOV for responsive sliders. (`preview`
keeps the real pixel size and optics — so blur and brightness look *exactly*
right — and just images a smaller patch of tissue for a few seconds.)

In [ ]:
SEED = 7
steps = []          # we append one StepSpec per stage, as we commit to it

# acq is created in Stage 1; these helpers read whatever it currently is.
def build(until=None):
    # full-resolution recording from the stages committed so far
    spec = Spec(acquisition=acq, seed=SEED, output=Output(save_intermediates=True), steps=list(steps))
    return simulate(spec, until=until)

def preview(extra=None, until=None, n_px=128, duration_s=3.0):
    # fast, small-FOV recording for slider exploration: same pixel size + optics,
    # just a smaller patch and fewer frames. `extra` is a step to try on top of
    # the committed ones without appending it.
    fast = acq.model_copy(update={
        "image_sensor": acq.image_sensor.model_copy(update={"n_px_height": n_px, "n_px_width": n_px}),
        "duration_s": duration_s,
    })
    trial = list(steps) + ([extra] if extra is not None else [])
    spec = Spec(acquisition=fast, seed=SEED, output=Output(save_intermediates=True), steps=trial)
    return simulate(spec, until=until)

## Stage 1 — How miniscope imaging works

Before committing to a specific scope, let's build intuition for the physics that
turns a fluorescent neuron into pixels. **This section is a sandbox:** it runs the
simulator's real optics, tissue, and sensor math on a few hand-placed cells, but
it is *completely independent* of the recording we build later — drag anything you
like, nothing here carries forward until we **commit a scope** at the bottom.

The image is shown in **green**, the way GCaMP fluorescence looks: a dark field
with bright cells.

**Left — the side view** (the picture you'd sketch on a whiteboard): the miniscope
at top, its **working distance** down to the focal plane, the **tissue** we image
through, and five cells spanning a range of depths around the focal plane.
**Right — what the image sensor sees.**

First, the **cell type** — a setting at the top of the next cell, not a slider.
`_MORPHOLOGY` chooses the GCaMP variant: **soma-targeted** GCaMP (SomaGCaMP /
riboGCaMP) confines the indicator to the cell body, so each cell is a clean blob;
**cytosolic** GCaMP (the usual GCaMP6/7/8, and the default here) also fills the
**proximal dendrites**, adding a few thin, fainter projections. Watch those
dendrites as you add tissue or push a cell off focus: the thin features are the
*first* thing scatter, defocus, and the noise floor erase, so usually only the
in-focus, shallow cell keeps them — exactly why resolving fine neurites through
tissue is so hard. (Change `_MORPHOLOGY` / `_N_DENDRITES` and re-run to compare.)

Then eight knobs, four families of physics:

- **Forming the image** — *NA* (light collection ∝ NA², so higher NA is mostly
  *brighter* here; its sharpening is tiny because diffraction is already
  *sub-pixel* — a miniscope is **pixel-limited**, not diffraction-limited),
  *magnification* (zooms the field of view) and *pixel pitch* (how coarsely the
  chip samples). These last two also set how much light lands on **each pixel**:
  a pixel collects flux over the object area it sees, `(pitch / magnification)²`,
  so higher mag *or* finer pitch spreads the same light over more pixels and dims
  every one of them — the **resolution-vs-SNR tradeoff**.
- **Depth & focus** — *tissue thickness* (how much tissue we image through: thin →
  bright & crisp, thick → dim & smeared by scatter + attenuation) and *focus* (the
  V4's tunable focus, **±150 µm**: a cell off the focal plane blurs, and one beyond
  reach can never be made sharp). The shaded band marks the **depth of field**,
  `≈ n·λ/NA²` — raise NA and watch it *narrow* (sharper but shallower focus). Note
  the asymmetry too: going *deeper* costs both focus *and* light, so deep cells
  fade fastest.
- **Field curvature** — *field curv*: a miniscope has no room for the lens elements
  that flatten the focal plane, so the in-focus surface is a shallow **bowl**, not a
  plane (the side view draws it): cells off the optical axis come into focus
  *shallower*, nearer the scope. The radius is large (real scopes ≈ 2–3 mm) so the
  bowl is gentle, but because the depth of field is only microns, even a small
  off-axis shift pushes edge cells out of focus — which is why the corners of a
  miniscope movie are softer and people keep their cells of interest centered. Dial
  the radius down to exaggerate it.
- **Detecting the image** — *exposure* and *read noise*: the sensor turns light
  into noisy counts, and a dim cell can sink below the noise floor. That "is it
  above the noise?" question is exactly what decides whether the analysis pipeline
  can ever recover a cell.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch, Rectangle
from scipy.ndimage import zoom
from minian.simulation.steps.cell import degrade_footprint, neuron_footprint

# A DISCONNECTED sandbox: real simulator physics on five hand-placed cells,
# independent of the scope we commit below. GCaMP-like LUT (black -> green).
GCAMP = LinearSegmentedColormap.from_list("gcamp", ["#000000", "#00b140", "#b6ffb6"])
WD_UM = 700.0          # Miniscope V4 nominal front working distance
CHIP_UM = 900.0        # sandbox sensor chip extent
_SOMA_UM, _EMISSION_NM, _MFP_UM, _BLUR_PER_UM = 6.0, 525.0, 100.0, 0.02
_N_TISSUE = 1.33        # tissue refractive index (DOF formula)
_PX_REF_UM = 5.0 / 4.0  # = default pitch/mag (1.25 um/px); per-pixel light ~ (px_um/this)^2

# Cell-type settings (not sliders -- edit here and re-run the cell to switch).
# Soma-targeted GCaMP confines signal to the body; cytosolic (the default here)
# also fills a few tapering proximal dendrites -- the thin features that scatter,
# defocus, and the noise floor erase first.
_MORPHOLOGY = "cytosolic"  # "soma" or "cytosolic"
_N_DENDRITES = 4
_DENDRITE_LEN_UM = 45.0    # proximal-dendrite length
_DENDRITE_WIDTH_UM = 3.0   # base width (tapers to a thread toward the tip)


def _dof_half_um(na):
    # in-focus half-depth; textbook DOF ~ n*lambda/NA^2 -> shrinks as NA rises
    return _N_TISSUE * (_EMISSION_NM / 1000.0) / na**2


# five cells spanning a depth band around the 700 um focal anchor, at distinct
# lateral spots so they never overlap. 1 = shallowest, 5 = deepest.
_CELLS = [
    {"name": "1", "y": -40.0, "x": -75.0, "offset": -50.0, "color": "#4daf4a"},
    {"name": "2", "y":  38.0, "x": -30.0, "offset": -25.0, "color": "#1f9e89"},
    {"name": "3", "y":  -8.0, "x":   8.0, "offset":   0.0, "color": "#377eb8"},
    {"name": "4", "y":  40.0, "x":  48.0, "offset":  27.0, "color": "#e6a700"},
    {"name": "5", "y": -34.0, "x":  80.0, "offset":  55.0, "color": "#e41a1c"},
]

# Generate each cell's sharp shape ONCE on a fixed fine um grid, so the shape is
# independent of the sensor sampling. Changing mag/pitch later only *rescales*
# these patches (same cells, zoomed -- never re-randomized); only editing the
# cell-type settings above rebuilds them.
# 0.5 um/px reference grid: finer than the sensor ever samples (object-space pixel
# ~1.25 um at the defaults) and on par with the diffraction-limited spot the optics
# can't beat anyway -- a 1p miniscope is pixel-limited, never diffraction-limited --
# so the reference holds the "true" shape without ever being the bottleneck.
_REF_PX_UM = 0.5
_PATCH_HALF_UM = 64.0  # half-extent of each cell's patch (covers soma + dendrites)
_ref_cache = {}        # (morphology, n_dendrites) -> list of fixed reference patches


def _ref_patches():
    key = (_MORPHOLOGY, _N_DENDRITES)
    if key not in _ref_cache:
        rng = np.random.default_rng(0)  # fixed seed: identical shapes every build
        n = int(round(2 * _PATCH_HALF_UM / _REF_PX_UM))
        c = (n - 1) / 2.0  # each cell sits at the center of its own patch
        _ref_cache[key] = [
            neuron_footprint(
                (n, n), (c, c), _SOMA_UM / _REF_PX_UM, 0.35, rng,
                morphology=_MORPHOLOGY, n_dendrites=_N_DENDRITES,
                dendrite_length_px=_DENDRITE_LEN_UM / _REF_PX_UM,
                dendrite_width_px=_DENDRITE_WIDTH_UM / _REF_PX_UM,
            )
            for _ in _CELLS
        ]
    return _ref_cache[key]


def _add_centered(canvas, patch, cy, cx):
    # Accumulate `patch` into `canvas` with the patch center aligned to (cy, cx),
    # clipping whatever falls off the canvas edge.
    ph, pw = patch.shape
    y0, x0 = int(round(cy - (ph - 1) / 2.0)), int(round(cx - (pw - 1) / 2.0))
    ys0, xs0 = max(y0, 0), max(x0, 0)
    ys1, xs1 = min(y0 + ph, canvas.shape[0]), min(x0 + pw, canvas.shape[1])
    if ys0 < ys1 and xs0 < xs1:
        canvas[ys0:ys1, xs0:xs1] += patch[ys0 - y0:ys1 - y0, xs0 - x0:xs1 - x0]


def _image_cells(na, magnification, pixel_pitch_um, tissue_thickness_um,
                 focus_offset_um, exposure, read_noise_e, field_curv_mm):
    # Build the five-cell scene with the simulator's real optics/tissue/sensor.
    optics = Optics(na=na, magnification=magnification, emission_nm=_EMISSION_NM,
                    field_curvature_radius_um=field_curv_mm * 1000.0)
    tissue = Tissue(scatter_mfp_um=_MFP_UM, scatter_blur_per_um=_BLUR_PER_UM)
    px_um = pixel_pitch_um / magnification
    n_px = int(np.clip(round(CHIP_UM / pixel_pitch_um), 48, 512))
    sensor = ImageSensor(n_px_height=n_px, n_px_width=n_px, pixel_pitch_um=pixel_pitch_um,
                         quantum_efficiency=0.7, read_noise_e=read_noise_e,
                         gain_adu_per_e=1.0, bit_depth=8)
    c = (n_px - 1) / 2.0
    focal_dist = WD_UM + focus_offset_um
    dof = _dof_half_um(na)
    optical = np.zeros((n_px, n_px))
    info = []
    for cell, patch in zip(_CELLS, _ref_patches()):
        dist = WD_UM + cell["offset"]                          # distance from scope
        path = max(tissue_thickness_um + cell["offset"], 0.0)  # tissue the light crosses
        # field curvature: off-axis cells focus shallower (no field flattener), so
        # each cell sees its own focal depth set by its radius from the optical axis
        r = np.hypot(cell["y"], cell["x"])
        focal_eff = focal_dist - optics.focal_curvature_shift_um(r)
        # the fixed shape, rescaled to the current pixel size (same cell, zoomed)
        sharp = np.clip(zoom(patch, _REF_PX_UM / px_um, order=1), 0.0, 1.0)
        cy, cx = c + cell["y"] / px_um, c + cell["x"] / px_um
        placed = np.zeros((n_px, n_px))
        _add_centered(placed, sharp, cy, cx)
        defocus = optics.defocus_sigma_um(dist, focal_eff)
        sigma_um = np.hypot(optics.diffraction_sigma_um,
                            np.hypot(tissue.scatter_sigma_um(path), defocus))
        gain = tissue.attenuation(path) * optics.collection_efficiency
        optical += degrade_footprint(placed, sigma_um / px_um, gain)
        info.append({**cell, "dist": dist, "in_focus": abs(dist - focal_eff) <= dof})
    # per-pixel light: a pixel integrates flux over its object area px_um^2 =
    # (pitch/mag)^2, so finer pitch OR higher mag dims each pixel (normalized to
    # the default px size, so the default state is unchanged).
    rng = np.random.default_rng(1)  # sensor shot/read noise (stable across redraws)
    counts = sensor.photons_to_counts(optical * exposure * (px_um / _PX_REF_UM) ** 2, rng)
    meta = dict(px_um=px_um, n_px=n_px, fov_um=n_px * px_um, focal_dist=focal_dist,
                surf=WD_UM - tissue_thickness_um, dof=dof, optics=optics)
    return counts, info, meta


# Build the figure ONCE; the fixed-range colorbar (0-255) is created once so it
# never accumulates when we redraw.
_fig1, (_ax1L, _ax1R) = plt.subplots(1, 2, figsize=(11.5, 4.8))
_fig1.subplots_adjust(left=0.07, right=0.87, wspace=0.30, top=0.84, bottom=0.12)
if hasattr(_fig1.canvas, "header_visible"):
    _fig1.canvas.header_visible = False
_cax1 = _fig1.add_axes([0.89, 0.12, 0.015, 0.72])
_sm1 = ScalarMappable(norm=Normalize(0, 255), cmap=GCAMP); _sm1.set_array([])
_fig1.colorbar(_sm1, cax=_cax1, label="ADC counts (8-bit)")

_sliders1 = dict(
    na=FloatSlider(min=0.1, max=0.6, step=0.02, value=0.30, description="NA"),
    magnification=FloatSlider(min=1.0, max=10.0, step=0.5, value=4.0, description="mag"),
    pixel_pitch_um=FloatSlider(min=1.0, max=8.0, step=0.2, value=5.0, description="pitch um"),
    tissue_thickness_um=FloatSlider(min=0.0, max=600.0, step=20.0, value=100.0, description="tissue um"),
    focus_offset_um=FloatSlider(min=-150.0, max=150.0, step=10.0, value=0.0, description="focus um"),
    exposure=FloatSlider(min=1000.0, max=20000.0, step=500.0, value=6000.0, description="exposure"),
    read_noise_e=FloatSlider(min=0.0, max=10.0, step=0.5, value=2.0, description="read noise"),
    field_curv_mm=FloatSlider(min=0.5, max=8.0, step=0.5, value=2.5, description="field curv mm"),
)


def explore_imaging():
    v = {k: s.value for k, s in _sliders1.items()}
    counts, info, meta = _image_cells(**v)
    axL, axR = _ax1L, _ax1R
    axL.clear(); axR.clear()
    # -- left: side view (depth = distance from scope, 0 at top) --
    axL.set_xlim(-150, 150); axL.set_ylim(-90, 900); axL.invert_yaxis()
    axL.add_patch(Rectangle((-55, -85), 110, 70, color="0.25"))
    axL.text(0, -50, "miniscope", color="w", ha="center", va="center", fontsize=9)
    axL.axhline(0, color="0.25", lw=1.5)
    surf = meta["surf"]
    axL.add_patch(Rectangle((-150, surf), 300, 900 - surf, color="#f0c9a8", alpha=0.55, zorder=0))
    axL.text(-145, surf + 22, "tissue", color="#a0522d", fontsize=9, va="top")
    axL.axhline(surf, color="#a0522d", ls=":", lw=1)
    axL.add_patch(FancyArrowPatch((-120, 0), (-120, WD_UM), arrowstyle="<->",
                                  mutation_scale=10, color="0.4", lw=1))
    axL.text(-128, WD_UM / 2, "WD 700 um", rotation=90, ha="right", va="center",
             color="0.4", fontsize=8)
    dof = meta["dof"]
    # field curvature: the in-focus surface is a shallow bowl (edges focus
    # shallower), not a flat plane. Draw its cross-section + the DOF band around it.
    xg = np.linspace(-150, 150, 121)
    opt = meta["optics"]
    fcurve = meta["focal_dist"] - np.array([opt.focal_curvature_shift_um(abs(x)) for x in xg])
    axL.fill_between(xg, fcurve - dof, fcurve + dof, color="#1f6fb2", alpha=0.16, zorder=1)
    axL.plot(xg, fcurve, color="#1f6fb2", ls="--", lw=1.4, zorder=2)
    axL.text(145, fcurve[-1], f"focal surface\n+/-{dof:.0f} um DOF", color="#1f6fb2",
             ha="right", va="top", fontsize=8)
    for ci in info:  # id number inside each dot (matches sensor panel); in-focus = white ring
        axL.scatter(ci["x"], ci["dist"], s=300, color=ci["color"], zorder=5,
                    edgecolor=("white" if ci["in_focus"] else "0.2"), linewidth=2.0)
        axL.text(ci["x"], ci["dist"], ci["name"], color="white", ha="center", va="center",
                 fontsize=9, weight="bold", zorder=6)
    axL.set(xlabel="lateral (um)", ylabel="distance from scope (um)",
            title="side view: scope -> tissue -> cells")
    # -- right: what the image sensor sees (colorbar is the fixed _cax1) --
    axR.imshow(counts, cmap=GCAMP, vmin=0, vmax=255, interpolation="nearest")
    for ci in info:
        cy = (meta["n_px"] - 1) / 2 + ci["y"] / meta["px_um"]
        cx = (meta["n_px"] - 1) / 2 + ci["x"] / meta["px_um"]
        if 0 <= cx < meta["n_px"] and 0 <= cy < meta["n_px"]:  # skip cells off the FOV
            axR.text(cx, cy - _SOMA_UM / meta["px_um"] - 4, ci["name"], color=ci["color"],
                     ha="center", fontsize=8, weight="bold")
    axR.set(title=f"image sensor: {meta['n_px']}x{meta['n_px']} px  |  {meta['px_um']:.2f} um/px"
                  f"  |  FOV {meta['fov_um']:.0f} um\nNA {v['na']:g}: brightness NA^2="
                  f"{meta['optics'].collection_efficiency:.3f}, diffraction sigma "
                  f"{meta['optics'].diffraction_sigma_um*1000:.0f} nm",
            xlabel="sensor px", ylabel="sensor px")
    _fig1.canvas.draw_idle()


interactive_panel(_sliders1, explore_imaging, _fig1.canvas)

**Commit the scope we'll simulate.** Everything above was intuition-building and is now behind us. Here we fix the *actual* scope the rest of the notebook uses — independent of whatever you dragged in the sandbox. The focal plane sits at 200 µm; we'll place cells around it next. *(These exact numbers are a placeholder we'll revisit when we tune for realism.)*

In [ ]:
acq = Acquisition(
    fps=20.0,
    duration_s=90.0,
    optics=Optics(na=0.18, magnification=3.0, emission_nm=525.0,
                  focal_plane_um=200.0, depth_of_field_um=15.0),
    image_sensor=ImageSensor(n_px_height=300, n_px_width=300, pixel_pitch_um=4.8, bit_depth=8),
    tissue=Tissue(scatter_mfp_um=100.0, scatter_blur_per_um=0.02),
)
print(f"FOV {acq.fov_um[0]:.0f} x {acq.fov_um[1]:.0f} um | {acq.pixel_size_um:.2f} um/px "
      f"| {acq.n_frames} frames @ {acq.fps:.0f} fps | focal plane {acq.optics.focal_plane_um:.0f} um")

## Stage 2 — Place the somata

Now we put cell bodies into the tissue volume — lateral position plus a **depth**
`z`. Depth is the quiet protagonist of this whole notebook: it decides how blurred
and how dim each cell ends up, and ultimately whether it is recoverable at all.

**Explore:** drag the density, the depth band (relative to the 200 µm focal
plane), and the soma size. The left panel is where cells sit (color = depth); the
right shows a few of their sharp, optics-free footprints (`A_planted`) — the ideal
target the analysis would love to recover.

In [ ]:
_DEPTH_MAX = 300.0  # fixed depth color scale, so the colorbar never changes

_fig2 = plt.figure(figsize=(10.5, 5.2))
if hasattr(_fig2.canvas, "header_visible"):
    _fig2.canvas.header_visible = False
_gs2 = _fig2.add_gridspec(2, 6, height_ratios=[3.0, 1.25], hspace=0.5, wspace=0.35,
                          left=0.08, right=0.84, top=0.9, bottom=0.1)
_ax2 = _fig2.add_subplot(_gs2[0, :])
_thumbs2 = [_fig2.add_subplot(_gs2[1, k]) for k in range(6)]
_cax2 = _fig2.add_axes([0.87, 0.46, 0.015, 0.42])
_sm2 = ScalarMappable(norm=Normalize(0, _DEPTH_MAX), cmap="viridis"); _sm2.set_array([])
_fig2.colorbar(_sm2, cax=_cax2, label="depth z (um)")

_sliders2 = dict(
    density_per_mm2=FloatSlider(min=100, max=2000, step=100, value=700, description="density/mm2"),
    depth_lo=FloatSlider(min=0, max=250, step=10, value=180, description="depth lo um"),
    depth_hi=FloatSlider(min=0, max=300, step=10, value=220, description="depth hi um"),
    soma_radius_um=FloatSlider(min=2, max=12, step=0.5, value=5, description="soma r um"),
)


def explore_placement():
    v = {k: s.value for k, s in _sliders2.items()}
    rec = preview(extra=PlaceSomata(density_per_mm2=v["density_per_mm2"], soma_radius_um=v["soma_radius_um"],
                                    depth_range_um=(v["depth_lo"], v["depth_hi"])),
                  until="place_somata")
    g = rec.ground_truth
    _ax2.clear()
    if g.n_units:
        z, y, x = g.centers_um[:, 0], g.centers_um[:, 1], g.centers_um[:, 2]
        _ax2.scatter(x, y, c=z, s=30, cmap="viridis", vmin=0, vmax=_DEPTH_MAX)
    _ax2.invert_yaxis()
    _ax2.set(title=f"{g.n_units} cells in this preview FOV (color = depth)",
             xlabel="x (um)", ylabel="y (um)")
    for k, tax in enumerate(_thumbs2):  # first few planted footprints (the ideal target)
        tax.clear(); tax.axis("off")
        if k < g.n_units:
            tax.imshow(g.A_planted[k], cmap="magma")
            tax.set_title(f"z={g.centers_um[k, 0]:.0f}", fontsize=8)
    _fig2.canvas.draw_idle()


interactive_panel(_sliders2, explore_placement, _fig2.canvas)

**Commit the somata.** Append the stage with our chosen values.

In [ ]:
steps.append(PlaceSomata(density_per_mm2=700.0, soma_radius_um=5.0, depth_range_um=(180.0, 220.0)))

rec = build(until="place_somata"); gt = rec.ground_truth
z = gt.centers_um[:, 0]
print(f"placed {gt.n_units} cells | depth {z.min():.0f}-{z.max():.0f} um "
      f"| focal plane at {acq.optics.focal_plane_um:.0f} um")
print("(which cells are in focus / detectable is decided at the optics stage, next)")

## Stage 3 — Calcium activity

Each cell now gets a spike train and the fluorescence it produces. Spikes come
from a 2-state (quiescent/active) gate driving a Poisson process; each spike adds
a calcium transient with a fast rise and slow decay,
`k(t) = e^{-t/τ_d} − e^{-t/τ_r}`. This rise/decay shape is exactly what
deconvolution later assumes.

**Explore:** the active firing rate and the decay time `τ_d`. Watch the traces:
faster rate → busier; longer `τ_d` → fatter, slower-falling transients. (The
sampling rule `τ_d · fps ≳ 1` must hold or the decay is unresolvable.)

In [ ]:
_fig3, _ax3 = plt.subplots(figsize=(9.5, 3.4))
_fig3.subplots_adjust(left=0.08, right=0.97, top=0.84, bottom=0.18)
if hasattr(_fig3.canvas, "header_visible"):
    _fig3.canvas.header_visible = False

_sliders3 = dict(
    active_rate_hz=FloatSlider(min=0.5, max=8.0, step=0.5, value=2.0, description="rate Hz"),
    tau_decay_s=FloatSlider(min=0.1, max=1.5, step=0.1, value=0.5, description="tau_d s"),
)


def explore_activity():
    v = {k: s.value for k, s in _sliders3.items()}
    rec = preview(extra=CellActivity(active_rate_hz=v["active_rate_hz"], tau_decay_s=v["tau_decay_s"]),
                  until="cell_activity", duration_s=20.0)
    g = rec.ground_truth
    t = np.arange(rec.spec.acquisition.n_frames) / rec.spec.acquisition.fps
    busiest = np.argsort(g.S.sum(axis=1))[-4:] if g.n_units else []
    _ax3.clear()
    for k, u in enumerate(busiest):
        _ax3.plot(t, g.C[u] + k * 1.2, lw=1.1)
        spk = np.where(g.S[u] > 0)[0]
        _ax3.plot(t[spk], np.full_like(spk, k * 1.2, dtype=float) - 0.2, "|", color="k", ms=6)
    _ax3.set(title=f"traces C (lines) + spikes S (ticks) | rate {v['active_rate_hz']} Hz, "
                   f"tau_d {v['tau_decay_s']}s", xlabel="time (s)", ylabel="cell (offset)")
    _fig3.canvas.draw_idle()


interactive_panel(_sliders3, explore_activity, _fig3.canvas)

**Commit the activity.** Append the stage; the cells now have traces.

In [ ]:
steps.append(CellActivity(active_rate_hz=2.0, tau_decay_s=0.5))
print("stages committed so far:", " -> ".join(s.kind for s in steps))

---

### Next: the cells become a movie

So far we have cells with positions, depths, and traces — but no image yet. The
next stages turn them into a movie and then degrade it the way the physical world
does: **optics** (blur + dim by depth), **render** (the first actual video!),
then **background, bleaching, motion, illumination, and sensor noise**. Those
stages produce real video at full resolution, so we'll decide *there* how to
balance slider responsiveness against fidelity — stage by stage.

*(More stages will be added below.)*